# 🎬 Cartoon-3D Short — renderer for a free Colab **T4**
Pulls the whole project from GitHub and renders it **here** (three.js inside Remotion on the T4's GPU) — nothing is rendered on your own machine.

**Before you run:** `Runtime ▸ Change runtime type ▸ T4 GPU`. If the repo is private, add a Colab **Secret** named `GITHUB_TOKEN` (key icon on the left, enable *Notebook access*) — or paste the token when asked (press Enter if the repo is public). The token is never written into the repo or this notebook.

Order: 1 params → 2 GPU check → 3 clone → 4 setup (≈4 min, once) → 5 GPU probe → 6 validate → 7 voice → 8 sound → 9 look-check stills → 10 render → 11 download.

In [ ]:
#@title 1 · Parameters
REPO        = "vidlgtm/testchike"  #@param {type:"string"}
BRANCH      = "main"               #@param {type:"string"}
PROJECT     = "bridge"             #@param {type:"string"}
QUALITY     = "1080p"              #@param ["preview", "1080p", "2k"]
VOICE_STEP  = True                 #@param {type:"boolean"}
STILLS      = "100,400,900"        #@param {type:"string"}
CHUNK       = 300                  #@param {type:"integer"}
CONCURRENCY = 2                    #@param {type:"integer"}
USE_DRIVE   = True                 #@param {type:"boolean"}
# preview = 540x960 (fast check) | 1080p = 1080x1920 | 2k = 1440x2560 (slowest)

In [ ]:
#@title 2 · GPU check
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out)
assert "T4" in out or "L4" in out or "A100" in out or "V100" in out, "No GPU runtime! Runtime > Change runtime type > T4 GPU"

In [ ]:
#@title 3 · Clone the repo (+ Google Drive for resumable output)
import os, getpass, subprocess
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
TOKEN = TOKEN or getpass.getpass("GitHub token (press Enter if the repo is public): ")
AUTH = f"x-access-token:{TOKEN}@" if TOKEN else ""
NAME = REPO.split("/")[-1]
os.chdir("/content")
if not os.path.isdir(NAME):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{AUTH}github.com/{REPO}.git"], check=True)
    subprocess.run(["git", "-C", NAME, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)  # strip the token
else:
    subprocess.run(["git", "-C", NAME, "pull", "--ff-only", f"https://{AUTH}github.com/{REPO}.git", BRANCH], check=True)
del TOKEN, AUTH
os.chdir(f"/content/{NAME}")

OUT = f"/content/{NAME}/out/{PROJECT}"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/{NAME}_renders/{PROJECT}"   # finished chunks survive a disconnect and are skipped on re-run
os.makedirs(OUT, exist_ok=True)
print("repo:", os.getcwd(), "\noutput:", OUT)

In [ ]:
#@title 4 · One-time setup (Node, ffmpeg, Chrome, NVIDIA Vulkan/EGL, npm, Python)
!bash colab/setup_colab.sh 2>&1 | tail -40

In [ ]:
#@title 5 · Find the GPU WebGL backend (must say hardware, NOT swiftshader)
!python tools/gpu_probe.py

In [ ]:
#@title 6 · Validate the template contract + TypeScript
!python tools/check_project.py {PROJECT}

In [ ]:
#@title 7 · Voice + word timings (edge-tts + faster-whisper on the GPU)
import os
if VOICE_STEP:
    !python tools/voice.py {PROJECT}
    !python tools/words_to_frames.py {PROJECT} | head -30
else:
    print("skipped (voice.wav + words.json already in projects/%s/voice)" % PROJECT)

In [ ]:
#@title 8 · Sound (synthesized SFX + score + mix; CPU, a few minutes)
!python audio_engine/mix.py projects/{PROJECT} --out {OUT}/audio 2>&1 | tail -25

In [ ]:
#@title 9 · Look-check stills (half size) — review before the long render
if STILLS.strip():
    !python tools/render.py {PROJECT} --stills {STILLS} --out {OUT}
    import glob
    from IPython.display import Image, display
    for p in sorted(glob.glob(f"{OUT}/stills/*.jpg")):
        print(p); display(Image(p, width=270))

In [ ]:
#@title 10 · RENDER (chunked + resumable; skips chunks already on Drive)
!python tools/render.py {PROJECT} --quality {QUALITY} --chunk {CHUNK} --concurrency {CONCURRENCY} --skip-audio --out {OUT}

In [ ]:
#@title 11 · Result: QA report, preview, download
import glob, os
from IPython.display import Video, Image, display
vids = sorted(glob.glob(f"{OUT}/{PROJECT}_{QUALITY}.mp4"))
print(vids)
if vids:
    print(f"{os.path.getsize(vids[0])/1e6:.1f} MB")
    display(Video(vids[0], embed=False, width=270))
    for s in sorted(glob.glob(f"{OUT}/qa/sheet_*.jpg"))[:2]: display(Image(s, width=600))
    if not USE_DRIVE:
        from google.colab import files; files.download(vids[0])
    else:
        print("The file is in your Google Drive:", vids[0])